# EEG Classification with Neural Network
Binary classification: Alzheimer's (A) vs Control (C)  
Features: Per-lead band power (delta, theta, alpha, beta) across 19 EEG leads → 76 features total  
Data pipeline mirrors `sample.ipynb`

## Imports

In [ ]:
import warnings

import mne
import mne.io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import welch

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

## Constants

In [ ]:
fs = 500  # sampling rate (Hz)
N_LEADS = 19
N_SUBJECTS = 65

## Data Load
Same loading logic as `sample.ipynb` — reads raw EEG, flattens to 1D per subject, trims to 1 minute.

In [ ]:
# Suppress boundary event warnings
warnings.filterwarnings(
    "ignore",
    message=".*boundary.*",
    category=RuntimeWarning,
    module="mne"
)

eeg_file_path_format = 'openneuro_data/sub-{number:03}/eeg/sub-{number:03}_task-eyesclosed_eeg.set'

rows = []

for n in range(1, N_SUBJECTS + 1):
    file_path = eeg_file_path_format.format(number=n)
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
    data = raw.to_data_frame()
    data.drop('time', axis=1, inplace=True)
    data = pd.Series(data.to_numpy().flatten()).iloc[:N_LEADS * fs * 60]  # 19 leads * 500 Hz * 60s
    data = data.reset_index(drop=True)
    rows.append(data)

X_raw = pd.DataFrame(rows)
print(f"Loaded {len(X_raw)} subjects")

## Load Labels

In [ ]:
labels_file_path = 'openneuro_data/participants.tsv'

Y_raw = pd.read_csv(labels_file_path, sep='\t')
Y_raw = Y_raw['Group'].iloc[:N_SUBJECTS]

print(Y_raw.value_counts())

## Feature Extraction
Welch PSD per lead → mean band power for delta, theta, alpha, beta.  
Result: 76 features per subject (19 leads × 4 bands).

In [ ]:
rows = []

for i in range(len(X_raw)):
    subject = X_raw.iloc[i]
    entry = pd.Series([i, Y_raw.iloc[i]], index=['Subject', 'Class'])

    for lead in range(N_LEADS):
        signal = subject[subject.index % N_LEADS == lead].to_numpy()
        freqs, psd = welch(signal, fs=fs)

        delta_power = np.mean(psd[(freqs >= 1)  & (freqs < 4)])
        theta_power = np.mean(psd[(freqs >= 4)  & (freqs < 8)])
        alpha_power = np.mean(psd[(freqs >= 8)  & (freqs < 13)])
        beta_power  = np.mean(psd[(freqs >= 13) & (freqs < 20)])

        entry[f'lead_{lead}_delta'] = delta_power
        entry[f'lead_{lead}_theta'] = theta_power
        entry[f'lead_{lead}_alpha'] = alpha_power
        entry[f'lead_{lead}_beta']  = beta_power

    rows.append(entry)

band_powers = pd.DataFrame(rows)
print(band_powers.shape)
band_powers.head()

## Prepare Features and Labels

In [ ]:
feature_cols = [c for c in band_powers.columns if c not in ['Subject', 'Class']]

X = band_powers[feature_cols].astype(float).values
y_raw = band_powers['Class'].values

le = LabelEncoder()
y = le.fit_transform(y_raw)  # A=0, C=1

print("Feature matrix shape:", X.shape)
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

## Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

## Build and Train Neural Network
Architecture: 76 → 128 → 64 → 32 → 1 (binary output)  
Uses StandardScaler inside a Pipeline (same pattern as SVM.ipynb).

In [ ]:
model = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        activation='relu',
        solver='adam',
        alpha=0.001,           # L2 regularization
        learning_rate_init=0.001,
        max_iter=500,
        early_stopping=True,   # hold out 10% of train for validation
        validation_fraction=0.1,
        n_iter_no_change=20,   # stop if val loss doesn't improve for 20 epochs
        random_state=42,
        verbose=False
    ))
])

model.fit(X_train, y_train)
print(f"Stopped at iteration: {model.named_steps['mlp'].n_iter_}")

## Evaluate on Test Set

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

## Cross-Validation (Stratified K-Fold)
Important with small dataset (n=65) — gives more reliable accuracy estimate than a single split.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

print("CV Fold Accuracies:", cv_scores)
print(f"Mean: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

## Training Loss Curve

In [ ]:
mlp = model.named_steps['mlp']

plt.figure(figsize=(9, 4))
plt.plot(mlp.loss_curve_, label='Training Loss')
if hasattr(mlp, 'validation_scores_'):
    # validation_scores_ is accuracy, not loss — plot on secondary axis
    ax2 = plt.twinx()
    ax2.plot(mlp.validation_scores_, color='orange', linestyle='--', label='Val Accuracy')
    ax2.set_ylabel('Validation Accuracy')
    ax2.legend(loc='lower right')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('MLP Training Curve')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Feature Importance (Permutation-Based)
MLP doesn't have built-in feature importances like Random Forest, so we use permutation importance — shuffle each feature and see how much accuracy drops.

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model, X_test, y_test,
    n_repeats=30,
    random_state=42,
    scoring='accuracy'
)

importances = pd.Series(result.importances_mean, index=feature_cols)
top_features = importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 6))
top_features.plot(kind='bar')
plt.title('Top 20 Features by Permutation Importance')
plt.ylabel('Mean Accuracy Decrease')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()